Extracción y Procesamiento Satelital (Sentinel-5P)
### Proyecto GeoVision-CLIP Cali | Fusión de Datos Espaciales

Este notebook documenta la ingesta de datos satelitales que servirán como "verdad espacial" para nuestro modelo de Machine Learning, permitiendo extrapolar la contaminación a zonas de Cali donde el DAGMA no tiene cobertura física.

## 1. El Satélite y el Sensor (Contexto Técnico)
* **Misión:** [Sentinel-5 Precursor (S5P)](https://sentinels.copernicus.eu/web/sentinel/copernicus/sentinel-5p), operado por la Agencia Espacial Europea (ESA) dentro del programa Copernicus.
* **Sensor:** **TROPOMI** (Tropospheric Monitoring Instrument). Es un espectrómetro que mapea la atmósfera global diariamente midiendo la radiación solar retrodispersada desde el espacio hacia la Tierra.
* **Resolución Espacial:** Alta resolución de **5.5 km x 3.5 km** por píxel, lo que permite diferenciar la contaminación entre el norte (Yumbo) y el sur (Pance) de Cali.

## 2. Naturaleza y Formato de los Datos
A diferencia de los sensores terrestres que entregan archivos `.csv` unidimensionales, Earth Engine maneja arreglos multidimensionales (Tensores):
* **Formato Nativo:** NetCDF (Network Common Data Form), optimizado en la nube como una `ImageCollection` de Google Earth Engine.
* **Unidades de Medida:** El satélite NO mide la concentración a nivel de calle ($\mu g/m^3$). Mide la **Densidad de Columna Troposférica**, expresada en **moles por metro cuadrado ($mol/m^2$)**. Esto representa la cantidad total de moléculas del gas que hay en un pilar de aire desde el suelo hasta la cima de la troposfera.

## 3. Protocolo de Acceso (Para el Equipo de Trabajo)
El siguiente bloque de código inicializa la API. **Atención compañeros:** Si es la primera vez que ejecutan este notebook en su sesión, el comando `ee.Authenticate()` abrirá una ventana emergente. Deben iniciar sesión con su cuenta de Google y otorgar los permisos para que el entorno de Colab pueda descargar las imágenes del proyecto.

## 4. Visualización de la Firma Atmosférica (2018 - 2022)
Para entender la dinámica de dispersión, generaremos un mosaico promediando todos los pases del satélite sobre Cali durante nuestra ventana de estudio (2018-2022). Esto elimina las nubes residuales y revela la "huella digital" térmica y química de la ciudad.

In [ ]:
import ee
import geemap

# 1. Rutina de Autenticación Segura para el equipo
try:
    # Intenta inicializar directamente (funciona si ya te autenticaste antes)
    ee.Initialize(project='proyecto3ia-494900')
    print("Conectado a Google Earth Engine exitosamente.")
except Exception as e:
    print(" Autenticación requerida. Sigue las instrucciones del navegador...")
    ee.Authenticate() # Lanza el pop-up para el compañero que entra por primera vez
    ee.Initialize(project='proyecto3ia-494900')
    print("Autenticación completada y conectado a GEE.")

In [12]:
# 1. Paleta de colores más limpia (sin colores oscuros al inicio para evitar manchas negras)
palette_cientifica = ['blue', 'cyan', 'green', 'yellow', 'red']

# 2. Ajustamos los límites basados en el diagnóstico para forzar el contraste
viz_o3 = {'min': 0.114, 'max': 0.116, 'palette': palette_cientifica}
viz_so2 = {'min': 0.00001, 'max': 0.00010, 'palette': palette_cientifica}
viz_no2 = {'min': 0.00001, 'max': 0.00005, 'palette': palette_cientifica}

# 3. Configurar el mapa
Map = geemap.Map(basemap='HYBRID')
Map.centerObject(cali_roi, 12)

# --- EL TRUCO ESTÁ AQUÍ ---
# Aplicamos .resample('bilinear') a cada capa para difuminar los cuadros gigantes
# y usamos opacity=0.5 para que sea semitransparente
Map.addLayer(o3_col.resample('bilinear').clip(cali_roi), viz_o3, 'Ozono (O3)', opacity=0.5)
Map.addLayer(so2_col.resample('bilinear').clip(cali_roi), viz_so2, 'Dióxido de Azufre (SO2)', opacity=0.5)
Map.addLayer(no2_col.resample('bilinear').clip(cali_roi), viz_no2, 'Dióxido de Nitrógeno (NO2)', opacity=0.5)

# 4. Añadir controles
Map.addLayerControl()
Map.add_colorbar(viz_no2, label="Densidad de NO2 (mol/m^2)", layer_name="Dióxido de Nitrógeno (NO2)")

print(" Mapa renderizado con suavizado bilineal (Heatmap).")
Map

 Mapa renderizado con suavizado bilineal (Heatmap).


Map(center=[3.424999763692953, -76.50000000000118], controls=(WidgetControl(options=['position', 'transparent_…

In [8]:
# =========================================================
# BLOQUE DIAGNÓSTICO: Buscando los valores reales de Cali
# =========================================================

# 1. Asegurar que las variables existen (re-ejecutamos la búsqueda para estar seguros)
# [Asumiendo que cali_roi, fecha_inicio, fecha_fin y las colecciones NO2, SO2, O3 fueron definidas arriba]

print(" Iniciando diagnóstico de datos invisibles...\n")

def imprimir_estadisticas_reales(nombre_gas, coleccion_cruda, imagen_reducida, roi):
    try:
        # Verificamos si hay datos en el catálogo primero
        conteo = coleccion_cruda.size().getInfo()

        # Calculamos estadísticas regionales promedio reales sobre Cali
        estadisticas = imagen_reducida.reduceRegion(
            reducer=ee.Reducer.minMax(),
            geometry=roi,
            scale=5000, # Escala aproximada de Sentinel-5P para un cálculo rápido
            maxPixels=1e6
        ).getInfo()

        # Identificamos los nombres de las bandas calculadas
        keys = list(estadisticas.keys())

        print(f" Estadísticas reales para {nombre_gas} en Cali (2018-2022):")
        print(f"   - Total de registros satelitales promediados: {conteo}")
        print(f"   - Valor Mínimo Real: {estadisticas.get(keys[0], 'N/A')}") # Normalmente min
        print(f"   - Valor Máximo Real: {estadisticas.get(keys[1], 'N/A')}") # Normalmente max
        print("-" * 50)

    except Exception as e:
        print(f" Error al obtener estadísticas para {nombre_gas}. Asegúrate de haber definido la ROI y fechas arriba: {e}")

# Ejecutamos el diagnóstico para los 3 gases (usando las variables definidas en la celda anterior)
# Necesitas pasarle la colección cruda (el ImageCollection completo) Y la imagen promediada (.mean())
# Para este diagnóstico, re-calcularemos promedios locales si es necesario
# pero asumiendo tu script anterior, usaremos o3_col, so2_col, no2_col

# [Nota: Si en tu celda anterior solo te quedaste con la imagen reducida '.mean()',
#  este diagnóstico necesitará re-filtrar las colecciones crudas primero.
#  Asumiremos que usaste el código robusto que te di antes.]

# Volvemos a definir las colecciones crudas solo para el conteo seguro
o3_raw = ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_O3').filterBounds(cali_roi).filterDate(fecha_inicio, fecha_fin).select('O3_column_number_density')
so2_raw = ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_SO2').filterBounds(cali_roi).filterDate(fecha_inicio, fecha_fin).select('SO2_column_number_density')
no2_raw = ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_NO2').filterBounds(cali_roi).filterDate(fecha_inicio, fecha_fin).select('tropospheric_NO2_column_number_density')

# Ejecutamos la impresión con las imágenes promediadas (que ya tienes definidas en tu map script)
imprimir_estadisticas_reales("Ozono (O3)", o3_raw, o3_col, cali_roi)
imprimir_estadisticas_reales("Dióxido de Azufre (SO2)", so2_raw, so2_col, cali_roi)
imprimir_estadisticas_reales("Dióxido de Nitrógeno (NO2)", no2_raw, no2_col, cali_roi)

print("\n Copia los valores que aparecen arriba y pégalos aquí para actualizar tus color_range en el script del mapa.")

 Iniciando diagnóstico de datos invisibles...

 Estadísticas reales para Ozono (O3) en Cali (2018-2022):
   - Total de registros satelitales promediados: 21841
   - Valor Mínimo Real: 0.11568071029149639
   - Valor Máximo Real: 0.11510902673838323
--------------------------------------------------
 Estadísticas reales para Dióxido de Azufre (SO2) en Cali (2018-2022):
   - Total de registros satelitales promediados: 21107
   - Valor Mínimo Real: 0.0001144128454920426
   - Valor Máximo Real: 2.7631489374270498e-05
--------------------------------------------------
 Estadísticas reales para Dióxido de Nitrógeno (NO2) en Cali (2018-2022):
   - Total de registros satelitales promediados: 23223
   - Valor Mínimo Real: 4.858363063866821e-05
   - Valor Máximo Real: 1.906946766728839e-05
--------------------------------------------------

 Copia los valores que aparecen arriba y pégalos aquí para actualizar tus color_range en el script del mapa.


## 5. Interpretación de Magnitudes Satelitales
A diferencia de los sensores del DAGMA que reportan en microgramos ($\mu g/m^3$), el sensor TROPOMI mide la **densidad de columna**. Los valores obtenidos en nuestro diagnóstico revelan magnitudes en el orden de $10^{-5}$ para gases industriales.

### Parámetros de Visualización Ajustados:
* **Ozono ($O_3$):** Presenta una densidad estable alrededor de los $0.115 \text{ mol/m}^2$. Hemos ajustado el rango entre 0.11 y 0.12 para resaltar variaciones térmicas sutiles.
* **Dióxido de Nitrógeno ($NO_2$):** Marcador de tráfico pesado. El rango se sitúa entre $0.00001$ y $0.00005 \text{ mol/m}^2$.
* **Dióxido de Azufre ($SO_2$):** Vinculado al corredor industrial de Yumbo. Se visualiza en un rango de $0.00001$ a $0.00012 \text{ mol/m}^2$.

La paleta de colores empleada es **'Magma'** (de negro a amarillo brillante), lo que permite identificar los "hotspots" de contaminación sobre la infraestructura urbana de Cali.

## 6. Impacto en la Salud Pública y Umbrales de Riesgo

Para contextualizar la gravedad de las concentraciones detectadas por la red terrestre y extrapoladas mediante el satélite Sentinel-5P, es necesario referenciar los límites de exposición que alteran la biología humana.

Las normativas se basan en las directrices de la Organización Mundial de la Salud (OMS) y la legislación ambiental colombiana (Resolución 2254 de 2017). Es importante notar que **los límites de salud pública se miden en concentración superficial (**µg/m³**)**, por lo que las densidades de columna satelital ($mol/m^2$) actúan como indicadores espaciales de riesgo, más no como medidas de exposición directa.

###  Umbrales Críticos por Contaminante

| Contaminante | Límite OMS (24 horas) | Límite Colombia (Res. 2254) | Impacto Biológico Principal |
| :--- | :--- | :--- | :--- |
| **Dióxido de Nitrógeno ($NO_2$)** | 25 µg/m³ | 100 µg/m³ (1 hora) | **Inflamación de las vías respiratorias.** Penetra profundamente en los pulmones. Exposición prolongada disminuye el desarrollo pulmonar en niños y aumenta el riesgo de infecciones respiratorias. |
| **Dióxido de Azufre ($SO_2$)** | 40 µg/m³ | 50 µg/m³ (24 horas) | **Irritación aguda y broncoconstricción.** Afecta gravemente a personas con asma. En el ambiente, es el precursor principal de la lluvia ácida, afectando cuerpos de agua y cultivos. |
| **Ozono Troposférico ($O_3$)** | 100 µg/m³ (8 horas) | 100 µg/m³ (8 horas) | **Estrés oxidativo celular.** Reduce drásticamente la función pulmonar y agrava enfermedades como el EPOC (Enfermedad Pulmonar Obstructiva Crónica). Produce ardor en los ojos y fatiga extrema al hacer ejercicio al aire libre. |

### El Efecto Cóctel (Multi-pollutant)
En escenarios urbanos e industriales (como el corredor Yumbo-Cali), la exposición rara vez ocurre con un solo gas aisladamente. La inhalación simultánea de $NO_2$ y $SO_2$ genera un efecto sinérgico que debilita las mucosas del tracto respiratorio, haciéndolas exponencialmente más vulnerables al daño oxidativo posterior causado por el pico fotoquímico del $O_3$ en horas de la tarde.